<a href="https://colab.research.google.com/github/imimjohnmiller-droid/caddy/blob/master/chess_godnumber_colab_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chess God-Number Weighted Training — ParaRNN (Fixed)

**Fixes over v1:**
1. **UCI tokenizer** — fixed 1968-token closed vocabulary, never produces `<unk>` on legal moves
2. **Hard game boundaries** — no chunk ever crosses two different games
3. **Smooth tanh weight curve** — god number signal preserved across full eval range, not collapsed by linear+cap
4. **Curriculum filters by weight threshold** — Stage 0 only sees decisive positions; later stages relax
5. **Validation split** — top-1 move accuracy on 1% holdout
6. **`torch.compile()`** — ~25% speedup on PyTorch 2.0+

**Setup:** `Runtime → Change runtime type → T4 GPU`

In [ ]:
# Cell 1 — Install
!apt-get update && apt-get install -y zstd ninja-build build-essential git 2>&1 | tail -5
!pip install torch chess tqdm numpy 2>&1 | tail -5
# ... rest of the cell remains the same ...import os
os.chdir('/content')
if not os.path.exists('/content/ml-pararnn-v2'):
    !git clone https://github.com/aminaallali/ml-pararnn-v2.git
%cd /content/ml-pararnn-v2
!MAX_JOBS=4 pip install -e . 2>&1 | tail -10
import torch
print(f'CUDA: {torch.cuda.is_available()} | GPU: {torch.cuda.get_device_name(0)}')
print(f'Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print('ParaRNN ready')

In [1]:

# Cell 2 — PGN path
import os
from pathlib import Path

PGN_ZST_URL = 'https://database.lichess.org/standard/lichess_db_standard_rated_2026-04.pgn.zst'
PGN_ZST_PATH = Path('/content/chess_data/lichess_2026-04.pgn.zst')
PGN_ZST_PATH.parent.mkdir(parents=True, exist_ok=True)

if not PGN_ZST_PATH.exists():
    print("Downloading compressed PGN (this may take 10-20 mins)...")
    !wget -O {PGN_ZST_PATH} "{PGN_ZST_URL}"
else:
    print(f'Compressed PGN ready ({PGN_ZST_PATH.stat().st_size/1e6:.1f} MB)')

--2026-05-23 22:54:30--  https://database.lichess.org/standard/lichess_db_standard_rated_2026-04.pgn.zst
Resolving database.lichess.org (database.lichess.org)... 141.95.66.62, 2001:41d0:700:5e3e::
Connecting to database.lichess.org (database.lichess.org)|141.95.66.62|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 29325351334 (27G) [application/octet-stream]
Saving to: ‘/content/chess_data/lichess_2026-04.pgn.zst’

       /content/che   0%[                    ]   5.68M  8.54MB/s               ^C


In [ ]:
# Cell 3 — Config
import math, random, re
from pathlib import Path
SMOKE_TEST = True
BASE_PATH = Path('/content/chess_training')
CKPT_PATH = BASE_PATH / 'checkpoints'
LOG_PATH  = BASE_PATH / 'logs'
for p in [BASE_PATH, CKPT_PATH, LOG_PATH]: p.mkdir(parents=True, exist_ok=True)
CONFIG = {
    'state_dim':    512,
    'n_layers':     6,
    'num_heads':    4,
    'lr':           6e-4,
    'warmup':       100,
    'total_steps':  5000 if not SMOKE_TEST else 20,
    'ckpt_every':   500,
    'gn_sharpness': 3.0,   # tanh knee in pawns; higher = sharper focus
    'gn_eval_w':    0.6,   # contribution of absolute eval
    'gn_jump_w':    0.4,   # contribution of eval jump (blunder/brilliancy)
    'weight_cap':   4.0,   # ceiling after scaling
    'min_elo':      2300,
    'val_frac':     0.01,
    'val_every':    250,
}
print('Config ready | smoke test:', SMOKE_TEST)

In [ ]:
# Cell 4 — UCI tokenizer  (FIXED: replaces SAN tokenizer)
#
# v1 BUG: SAN vocab was built by sampling 5000 games.
# Rare but valid moves (e.g. 'Raa1', 'e8=N') could be absent
# and mapped to <unk>, allowing the model to emit <unk> as a move.
#
# FIX: UCI has a closed vocabulary of exactly 1968 move strings
# covering every legal move. No sampling, no <unk> on legal moves.
import chess

def build_uci_vocab():
    promo = ['q','r','b','n']
    moves = set()
    for fr in chess.SQUARES:
        for to in chess.SQUARES:
            if fr == to: continue
            fn, tn = chess.square_name(fr), chess.square_name(to)
            moves.add(fn+tn)
            if (chess.square_rank(fr)==6 and chess.square_rank(to)==7) or \
               (chess.square_rank(fr)==1 and chess.square_rank(to)==0):
                for p in promo: moves.add(fn+tn+p)
    SPECIAL = ['<pad>','<s>','</s>']
    tok2id = {t:i for i,t in enumerate(SPECIAL)}
    for m in sorted(moves):
        if m not in tok2id: tok2id[m] = len(tok2id)
    return tok2id, {v:k for k,v in tok2id.items()}

tok2id, id2tok = build_uci_vocab()
VOCAB_SIZE = len(tok2id)
PAD_ID, BOS_ID, EOS_ID = tok2id['<pad>'], tok2id['<s>'], tok2id['</s>']
print(f'UCI vocab: {VOCAB_SIZE} tokens | sample: {list(tok2id)[3:8]}')

In [ ]:

# Cell 5 — PGN parser + God Number weight formula (MODIFIED FOR STREAMING)
import chess.pgn
import subprocess
import math
import re

def parse_eval_comment(comment):
    m = re.search(r'\[%eval\s+([#]?[+-]?\d+(?:\.\d+)?)\]', comment)
    if not m: return None
    val = m.group(1)
    if val.startswith('#'):
        n = abs(int(val[1:]))
        sign = 1 if int(val[1:]) > 0 else -1
        return sign * max(100, 3000 - n * 100)
    return float(val) * 100.0

def compute_move_weight(eval_cp, prev_eval_cp=0.0, sharpness=3.0, eval_w=0.6, jump_w=0.4, cap=4.0):
    abs_p = abs(eval_cp) / 100.0
    jmp_p = abs(eval_cp - prev_eval_cp) / 100.0
    combined = eval_w * math.tanh(abs_p / sharpness) + jump_w * math.tanh(jmp_p / sharpness)
    return 1.0 + (cap - 1.0) * combined

def parse_games(pgn_zst_path, min_elo=None, max_games=None, require_evals=False):
    count = 0
    # STREAMING FIX: Decompress on-the-fly to save Colab disk space
    proc = subprocess.Popen(['zstd', '-dc', str(pgn_zst_path)], stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)

    try:
        while True:
            game = chess.pgn.read_game(proc.stdout)
            if game is None: break
            if min_elo:
                try:
                    if int(game.headers.get('WhiteElo','0')) < min_elo or \
                       int(game.headers.get('BlackElo','0')) < min_elo: continue
                except ValueError: continue

            board = game.board()
            uci_moves, weights, prev_eval, has_eval = [], [], 0.0, False
            node = game.next()
            while node is not None:
                uci_moves.append(node.move.uci())
                ev = parse_eval_comment(node.comment)
                if ev is not None:
                    has_eval = True
                    w = compute_move_weight(ev, prev_eval, CONFIG['gn_sharpness'], CONFIG['gn_eval_w'], CONFIG['gn_jump_w'], CONFIG['weight_cap'])
                    prev_eval = ev
                else:
                    w = 1.0  # FALLBACK: Default weight since Lichess dumps lack evals
                weights.append(w)
                board.push(node.move)
                node = node.next()

            # EVAL FIX: Disabled requirement so games aren't skipped
            if require_evals and not has_eval: continue
            if len(uci_moves) < 10: continue

            yield uci_moves, weights, game.headers.get('Result',' ')
            count += 1
            if max_games and count >= max_games: break
    finally:
        proc.terminate()

    print(f'Parsed {count} games')

if SMOKE_TEST:
    # Ensure you use the new ZST path here
    g = next(parse_games(PGN_ZST_PATH, min_elo=CONFIG['min_elo'], max_games=1))
    print(f'Game: {len(g[0])} moves | result: {g[2]}')
    for i in range(min(6,len(g[0]))): print(f'  {g[0][i]:6}  w={g[1][i]:.3f}')

In [ ]:
# Cell 6 — Data stream with hard game-boundary enforcement  (FIXED)
#
# v1 BUG: games were concatenated into a flat buffer and sliced into
# seq_len chunks. A chunk could span game A's end + game B's start,
# training the model to predict game B moves as continuations of game A.
#
# FIX: every sequence window is entirely within ONE game.
# Games shorter than seq_len are right-padded (PAD_ID, weight=0).
# Games longer than seq_len are split into non-overlapping windows,
# all within the same game.
#
# CURRICULUM FILTER: min_weight_threshold skips windows whose
# mean weight is below the threshold, so early stages only see
# high-signal (tactically decisive) positions.

import torch
import random

def encode_game(uci_moves, weights):
    ids = [BOS_ID] + [tok2id.get(m, PAD_ID) for m in uci_moves] + [EOS_ID]
    wts = [1.0]   + list(weights)                                 + [0.0]
    return ids, wts

def build_stream(pgn_path, tok2id, seq_len, batch_size, seed=42,
                 min_weight_threshold=0.0, val_frac=0.0, is_val=False):
    rng = random.Random(seed)
    bx, by, bw = [], [], []

    def game_source():
        while True:
            for uci_moves, weights, result in parse_games(
                pgn_path,
                min_elo=CONFIG['min_elo'] if CONFIG['min_elo'] > 0 else None
            ):
                in_val = (hash(tuple(uci_moves[:4])) % 1000) < int(val_frac * 1000)
                if is_val != in_val: continue
                yield uci_moves, weights
            print('Epoch complete, restarting...')

    for uci_moves, weights in game_source():
        ids, wts = encode_game(uci_moves, weights)
        total = len(ids)
        for start in range(0, total - 1, seq_len):
            end = start + seq_len + 1
            c_ids = ids[start:end]
            c_wts = wts[start:end]
            pad = (seq_len + 1) - len(c_ids)
            c_ids += [PAD_ID] * pad
            c_wts += [0.0]    * pad
            x_seq, y_seq, w_seq = c_ids[:-1], c_ids[1:], c_wts[1:]
            if min_weight_threshold > 0:
                valid_w = [w for w,y in zip(w_seq,y_seq) if y != PAD_ID]
                if not valid_w or sum(valid_w)/len(valid_w) < min_weight_threshold: continue
            bx.append(x_seq); by.append(y_seq); bw.append(w_seq)
            if len(bx) == batch_size:
                x = torch.tensor(bx, dtype=torch.long)
                y = torch.tensor(by, dtype=torch.long)
                w = torch.tensor(bw, dtype=torch.float32)
                w = w.masked_fill(y == PAD_ID, 0.0)
                yield x, y, w
                bx, by, bw = [], [], []

print('Data stream ready (game-boundary enforced)')

In [ ]:
# Cell 7 — Model
import torch.nn as nn, torch.nn.functional as F
from pararnn.rnn_cell.gru_diag_mh import GRUDiagMH, GRUDiagMHConfig
from pararnn.parallel_reduction.parallel_reduction import NewtonConfig

class ChessModel(nn.Module):
    def __init__(self, vocab_size, state_dim, n_layers, num_heads=4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, state_dim, padding_idx=PAD_ID)
        nn.init.normal_(self.embedding.weight, std=0.02)
        self.embedding.weight.data[PAD_ID].zero_()
        ncfg = NewtonConfig(max_its=3, tol=1e-4)
        self.gru_layers = nn.ModuleList([
            GRUDiagMH(GRUDiagMHConfig(
                state_dim=state_dim, input_dim=state_dim,
                device='cuda', dtype=torch.bfloat16, mode='parallel_fused',
                newton_config=ncfg, num_heads=num_heads,
                nonlin_update='sigmoid', nonlin_reset='sigmoid', nonlin_state='tanh',
            )) for _ in range(n_layers)])
        self.norms      = nn.ModuleList([nn.LayerNorm(state_dim, dtype=torch.bfloat16) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(state_dim, dtype=torch.bfloat16)
        self.lm_head    = nn.Linear(state_dim, vocab_size, bias=False)
        self.lm_head.weight = self.embedding.weight
        assert hasattr(self.gru_layers[0]._get_impl_type(), 'fused_parallel_forward')

    def forward(self, x):
        x = self.embedding(x)
        for gru, norm in zip(self.gru_layers, self.norms):
            x = x + gru(norm(x))
        return self.lm_head(self.final_norm(x))

device = 'cuda'
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32  = True
model = ChessModel(VOCAB_SIZE, CONFIG['state_dim'], CONFIG['n_layers'], CONFIG['num_heads']).to(device)
try:
    model = torch.compile(model)
    print('torch.compile() enabled (~25% speedup)')
except Exception as e:
    print(f'torch.compile() unavailable: {e}')
print(f'Params: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.1f}M')

In [ ]:

# Cell 8 — Training loop
import torch
import torch.nn.functional as F  # <-- Added for safety
from torch.utils.tensorboard import SummaryWriter
import math

def weighted_ce_loss(logits, targets, weights):
    loss = F.cross_entropy(logits.view(-1,VOCAB_SIZE), targets.view(-1), reduction='none').view_as(targets)
    w = weights.to(logits.device)
    return (loss * w).sum() / w.sum().clamp_min(1.0)

@torch.no_grad()
def move_accuracy(logits, targets):
    mask = targets != PAD_ID
    if not mask.any(): return float('nan')
    return (logits.argmax(-1)[mask] == targets[mask]).float().mean().item()

# Curriculum: (end_step, seq_len, batch_size, grad_accum, min_weight_threshold)
# UPDATED: min_weight_threshold set to 0.0 for all stages due to missing evals
if SMOKE_TEST:
    curriculum = [(20, 64, 2, 1, 0.0)]
else:
    curriculum = [
    ( 500, 128, 8, 4, 2.5),  # Stage 0: decisive positions only
    (1500, 256, 8, 4, 1.5),  # Stage 1: include moderate positions
    (5000, 256, 4, 8, 0.0),  # Stage 2: all positions
]

TOTAL_STEPS = curriculum[-1][0]
MAX_LR, MIN_LR, WARMUP = CONFIG['lr'], CONFIG['lr']/10, CONFIG['warmup']

def lr_at(step):
    if step < WARMUP: return MAX_LR*(step+1)/WARMUP
    p = (step-WARMUP)/max(1,TOTAL_STEPS-WARMUP)
    return MIN_LR + 0.5*(MAX_LR-MIN_LR)*(1+math.cos(math.pi*min(1.0,p)))

opt = torch.optim.AdamW(model.parameters(), lr=MAX_LR, betas=(0.9,0.95), weight_decay=0.1, eps=1e-8)
writer = SummaryWriter(str(LOG_PATH))

global_step = 0
last_ckpt = CKPT_PATH / 'last.pt'
if last_ckpt.exists() and not SMOKE_TEST:
    ck = torch.load(last_ckpt, map_location=device)
    (model._orig_mod if hasattr(model,'_orig_mod') else model).load_state_dict(ck['model'])
    opt.load_state_dict(ck['opt'])
    global_step = ck['step']
    print(f'Resumed from step {global_step}')

# UPDATED: Using PGN_ZST_PATH
val_stream = build_stream(PGN_ZST_PATH, tok2id, 256, 8,
                          val_frac=CONFIG['val_frac'], is_val=True)

@torch.no_grad()
def run_validation(n=20):
    model.eval()
    ls, ac, k = 0.0, 0.0, 0
    for _ in range(n):
        try: x,y,w = next(val_stream)
        except StopIteration: break
        x,y,w = x.to(device),y.to(device),w.to(device)
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            lg = model(x)
        ls += weighted_ce_loss(lg,y,w).item()
        ac += move_accuracy(lg,y)
        k += 1
    model.train()
    return (ls/k, ac/k) if k else (None, None)

cur_stage = -1; stream = None
accum_loss = 0.0; micro_count = 0
model.train()
print('Training start')

while global_step < TOTAL_STEPS:
    stage = next(i for i,(end,*_) in enumerate(curriculum) if global_step < end)
    if stage != cur_stage:
        cur_stage = stage
        _end, seq_len, bs, accum_steps, min_w = curriculum[stage]
        print(f'Stage {stage}: seq={seq_len} bs={bs} accum={accum_steps} min_w={min_w}')
        # UPDATED: Using PGN_ZST_PATH
        stream = build_stream(PGN_ZST_PATH, tok2id, seq_len, bs,
                              seed=42+global_step, min_weight_threshold=min_w,
                              val_frac=CONFIG['val_frac'], is_val=False)
    x,y,w = next(stream)
    x,y,w = x.to(device,non_blocking=True), y.to(device,non_blocking=True), w.to(device,non_blocking=True)
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        logits = model(x)
        loss = weighted_ce_loss(logits,y,w) / accum_steps
    if not torch.isfinite(loss):
        print(f'NaN at step {global_step}, skipping')
        opt.zero_grad(set_to_none=True); accum_loss=0.0; micro_count=0; global_step+=1; continue
    loss.backward()
    accum_loss += loss.item(); micro_count += 1
    if micro_count < accum_steps: continue
    lr = lr_at(global_step)
    for pg in opt.param_groups: pg['lr'] = lr
    gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step(); opt.zero_grad(set_to_none=True)
    if global_step % 10 == 0:
        al = accum_loss * accum_steps
        print(f'step {global_step:>6} | loss {al:.4f} | ppl {math.exp(min(20,al)):.1f} | lr {lr:.2e} | gnorm {gnorm:.2f} | avg_w {w.mean():.2f}')
        writer.add_scalar('train/loss', al, global_step)
        writer.add_scalar('train/lr', lr, global_step)
        writer.add_scalar('train/avg_weight', w.mean().item(), global_step)
    if global_step > 0 and global_step % CONFIG['val_every'] == 0:
        vl, va = run_validation()
        if vl: print(f'  val loss={vl:.4f} | move_acc={va*100:.1f}%'); writer.add_scalar('val/loss',vl,global_step); writer.add_scalar('val/move_acc',va,global_step)
    if global_step > 0 and global_step % CONFIG['ckpt_every'] == 0:
        tgt = model._orig_mod if hasattr(model,'_orig_mod') else model
        tmp = CKPT_PATH/'last.pt.tmp'
        torch.save({'step':global_step+1,'model':tgt.state_dict(),'opt':opt.state_dict(),'config':CONFIG},tmp)
        tmp.replace(last_ckpt)
        print(f'checkpoint @ {global_step}')
    global_step += 1; accum_loss = 0.0; micro_count = 0

tgt = model._orig_mod if hasattr(model,'_orig_mod') else model
torch.save({'step':global_step,'model':tgt.state_dict(),'config':CONFIG}, CKPT_PATH/'final.pt')
writer.close(); print('Training complete!')

In [ ]:
# Cell 9 — Save to Drive
from google.colab import drive
drive.mount('/content/drive')
!cp -r {BASE_PATH} /content/drive/MyDrive/
print('Saved to Google Drive')